In [1]:
# Deep Learning framework
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchsummary import summary

# Audio processing
import torchaudio
import torchaudio.transforms as T
import librosa

# Pre-trained image models
# import timm

# Play the audio in Jupyter notebook
from IPython.display import Audio
import pandas as pd
import os
import numpy as np

from scripts.dataset import AudioDataset
from models.cnn_tutorial import CNNNetworkTutorial

if torch.cuda.is_available():
    DEVICE = "cuda"
else:
    DEVICE = "cpu"
print(DEVICE)

cuda


In [2]:
def create_data_loader(train_data, batch_size):
    train_dataloader = DataLoader(train_data, batch_size=batch_size)
    return train_dataloader


def train_single_epoch(model, data_loader, loss_fn, optimizer):
    running_loss = 0.
    last_loss = 0.
    for i, inputs, labels in enumerate(data_loader):
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)

        # Zero your gradients for every batch!
        optimizer.zero_grad()

        # Make predictions for this batch
        outputs = model(inputs)

        # Compute the loss and its gradients
        loss = loss_fn(outputs, labels)
        loss.backward()

        # Adjust learning weights
        optimizer.step()

        # Gather data and report
        running_loss += loss.item()

        if i % 1000 == 999:
            last_loss = running_loss / 1000 # loss per batch
            print('  batch {} loss: {}'.format(i + 1, last_loss))
            running_loss = 0.

    return last_loss


def train(model, data_loader, loss_fn, optimizer, epochs):
    best_vloss = 1_000_000.
    for i in range(epochs):
        print(f"Epoch {i+1}")
        train_single_epoch(model, data_loader, loss_fn, optimizer)
        print("---------------------------")
    print("Finished training")


    # Initializing in a separate cell so we can easily add more epochs to the same run
    epoch_number = 0

    EPOCHS = 5

    best_vloss = 1_000_000.

    for epoch in range(EPOCHS):
        print(f'EPOCH {epoch_number + 1}:')

        # Make sure gradient tracking is on, and do a pass over the data
        model.train(True)
        avg_loss = train_single_epoch(model, data_loader, loss_fn, optimizer)

        # We don't need gradients on to do reporting
        model.train(False)

        running_vloss = 0.0
        for i, vdata in enumerate(validation_loader):
            vinputs, vlabels = vdata
            voutputs = model(vinputs)
            vloss = loss_fn(voutputs, vlabels)
            running_vloss += vloss

        avg_vloss = running_vloss / (i + 1)
        print('LOSS train {} valid {}'.format(avg_loss, avg_vloss))

        # Log the running loss averaged per batch
        # for both training and validation
        writer.add_scalars('Training vs. Validation Loss',
                        { 'Training' : avg_loss, 'Validation' : avg_vloss },
                        epoch_number + 1)
        writer.flush()

        # Track best performance, and save the model's state
        if avg_vloss < best_vloss:
            best_vloss = avg_vloss
            model_path = 'model_{}_{}'.format(timestamp, epoch_number)
            torch.save(model.state_dict(), model_path)

        epoch_number += 1

In [3]:
AUDIO_DIR = "audios/labeled/"

dict_genres = {'positive': 0, 'negative': 1}

reverse_map = {v: k for k, v in dict_genres.items()}
print(reverse_map)

{0: 'positive', 1: 'negative'}


In [4]:
data = []

for label in dict_genres.keys():
    for folder in os.listdir(AUDIO_DIR + label):
        for file in os.listdir(AUDIO_DIR + label + "/" + folder):
            file_path = AUDIO_DIR + label + "/" + folder + "/" + file
            positive = int(label == "positive")
            negative = int(label == "negative")
            data.append((file_path, positive,
                        negative))

file_path, positive, negative = zip(*data)
df = pd.DataFrame({"file_path": file_path, "positive": positive, "negative": negative})

print(df.head(5))

                                 file_path  positive  negative
0  audios/labeled/positive/1001/100100.wav         1         0
1  audios/labeled/positive/1002/100200.wav         1         0
2  audios/labeled/positive/1003/100300.wav         1         0
3  audios/labeled/positive/1003/100301.wav         1         0
4  audios/labeled/positive/1003/100302.wav         1         0


In [5]:
BATCH_SIZE = 1
EPOCHS = 5
LEARNING_RATE = 0.001
SAVE = False

dataset = AudioDataset(df)

train_dataloader = create_data_loader(dataset, BATCH_SIZE)

# construct model and assign it to device
model = CNNNetworkTutorial().to(DEVICE)
summary(model.cuda(), (1, 64, 44))

# initialise loss funtion + optimiser
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(),
                                lr=LEARNING_RATE)

# train model
train(model, train_dataloader, loss_fn, optimizer, EPOCHS)

# save model
if SAVE:
    torch.save(model.state_dict(), "feedforwardnet.pth")
    print("Trained feed forward net saved at feedforwardnet.pth")

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Conv2d-1           [-1, 16, 66, 46]             160
              ReLU-2           [-1, 16, 66, 46]               0
         MaxPool2d-3           [-1, 16, 33, 23]               0
            Conv2d-4           [-1, 32, 35, 25]           4,640
              ReLU-5           [-1, 32, 35, 25]               0
         MaxPool2d-6           [-1, 32, 17, 12]               0
            Conv2d-7           [-1, 64, 19, 14]          18,496
              ReLU-8           [-1, 64, 19, 14]               0
         MaxPool2d-9             [-1, 64, 9, 7]               0
           Conv2d-10           [-1, 128, 11, 9]          73,856
             ReLU-11           [-1, 128, 11, 9]               0
        MaxPool2d-12            [-1, 128, 5, 4]               0
          Flatten-13                 [-1, 2560]               0
           Linear-14                   